# 05 · Training
64px smoke run -> your configured run. Training is measured in kimg. The EMA generator is the inference generator.

In [ ]:
import os, sys
# ---- platform auto-detect: the same notebook runs on Colab and Kaggle ----
PLATFORM = "kaggle" if os.path.exists("/kaggle/input") else "colab"
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
if (PLATFORM == "colab" or PLATFORM == "kaggle") and not os.path.exists("src"):
    import subprocess
    subprocess.run(["git", "clone", "https://github.com/Ravikishore710/styleforge3-T.git"], check=True)
    os.chdir("styleforge3-T")
sys.path.insert(0, os.path.abspath("."))
!pip install -q -r requirements.txt
import tensorflow as tf
print("platform:", PLATFORM, "| TF:", tf.__version__,
      "| GPU:", tf.config.list_physical_devices("GPU"))
# Kaggle: enable GPU (Settings -> Accelerator -> GPU P100) and add the FFHQ
# dataset to /kaggle/input, or run scripts/prepare_ffhq.py --source folder.

## Smoke run (64px, minutes)
Verifies AMP, lazy R1, EMA, checkpointing and logging end to end.

In [ ]:
!python scripts/train.py --config configs/ffhq_64.yaml

## Watch training

In [ ]:
from src.visualization.training_curves import plot_log
plot_log("outputs/logs/log.jsonl", "outputs/logs")

In [ ]:
from src.config import load_config
from src.generator.generator import Generator
from src.inference.sampling import generate, save_grid, seeded_z
from src.training.checkpoint import CheckpointManager
import tensorflow as tf
cfg = load_config('configs/ffhq_64.yaml')
G_ema = Generator(cfg)
_ = G_ema(tf.zeros([1, int(cfg['z_dim'])]))
ckpt = CheckpointManager(cfg['output_dir'], G_ema, G_ema, G_ema, None, None,
                         tf.Variable(0, dtype=tf.int64), tf.Variable(0, dtype=tf.int64))
ckpt.restore(); ckpt.restore_ema_into(G_ema)
z = seeded_z(16, cfg['z_dim'], 0)
print(save_grid(generate(G_ema, z, noise_mode='const'),
                'outputs/samples/during_training.png'))

## Real run
```
!python scripts/train.py --config configs/ffhq_256.yaml
```
On Kaggle keep the session alive, or reconnect and rerun with `--resume`: checkpoints snap every `snap_kimg`.